# chat

> One conversation, whichever backend is behind it.

`Chat` holds the parts of a conversation that do not depend on the transport: the history, the options, the callbacks, and the cancel flag. A backend subclasses it, supplies a way to send one message, and inherits everything else.

`Chat('gpt-5.1')` returns the right subclass — dispatch happens in `__new__`, so `isinstance(c, Chat)` still holds. `chat()` is the same thing as a plain function, which is easier to read the signature of and easier to wrap.

In [ ]:
#| default_exp chat

In [ ]:
#| export
import asyncio, json, os, threading
from urllib.request import Request, urlopen
from dataclasses import replace
from fastcore.all import L, GetAttr, SaveReturn, asave_iter, patch
from urai.core import UsageStats, Resp, resp_text, run_cbs
from urai.msgs import mk_content, mk_msg, mk_msgs, is_media
from urai.opts import (RUNTIMES, ChatOpts, ModelSpec, split_runtime, resolve_runtime,
                       load_ref, set_dotted, turn_kw)

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from urai.opts import Runtime, register_runtime, resolve
from urai.core import ChatCallback
from urai.msgs import ToolCall

## When the window fills

Backends report an overflow as whatever exception their engine raised, so there is nothing structured to match on. `is_ctx_error` guesses from the message, and falls back to asking whether the window is in fact full.

In [ ]:
#| export
budget_msg_ = 'Tool-call budget exceeded; no more tools will run this turn.'
cancel_msg_ = 'Not run: the user stopped this turn.'
cancelled_reply_ = '(stopped by the user)'

class ContextWindowExceededError(RuntimeError):
    "A turn could not finish: the context window filled and recovery also failed."

_ctx_err_markers = ('context window', 'context length', 'max number of tokens', 'out of bounds',
                    'kv cache', 'n_ctx', 'exceed context', 'available state entries')

def is_ctx_error(chat, e):
    "Best effort: does `e`, from a backend step, look like the context window filling up?"
    if any(m in str(e).lower() for m in _ctx_err_markers): return True
    try: return bool(chat.ctx_limit) and chat.pct_full >= 1
    except Exception: return False

In [ ]:
class _Fake: ctx_limit, pct_full = 0, 0
test_eq(is_ctx_error(_Fake(), ValueError('n_ctx exceeded')), True)
test_eq(is_ctx_error(_Fake(), ValueError('KV cache is full')), True)   # matched case-insensitively
test_eq(is_ctx_error(_Fake(), ValueError('connection reset')), False)

In [ ]:
class _Full: ctx_limit, pct_full = 8192, 1.0
test_eq(is_ctx_error(_Full(), ValueError('something else')), True)     # the window really is full
class _Broken:
    ctx_limit = 8192
    @property
    def pct_full(self): raise RuntimeError('no idea')
test_eq(is_ctx_error(_Broken(), ValueError('unclear')), False)         # cannot tell, so no

## Finding a backend

The registry holds a dotted path, so this is the first point at which a backend module is imported. An optional runtime that is not installed says how to install it; a required one raises as it is.

In [ ]:
#| export
def get_runtime(nm):
    "The `Chat` subclass for runtime `nm`, importing the backend module lazily."
    rt = RUNTIMES[nm]
    try: return load_ref(rt.cls)
    except ImportError as e:
        if not rt.optional: raise
        raise ImportError(f'The {nm!r} runtime is unavailable ({e}). '
                          f'Install it with: {rt.install}') from None

In [ ]:
register_runtime(Runtime('missing', 'no_such_module_at_all.X', optional=True))
test_fail(lambda: get_runtime('missing'), contains='Install it with')
register_runtime(Runtime('required', 'no_such_module_at_all.X'))
test_fail(lambda: get_runtime('required'), contains='no_such_module_at_all')
del RUNTIMES['missing'], RUNTIMES['required']

## The chat

A backend supplies `_send`, and gets the rest. Everything configurable lives on `self.opts`, and the settings the loop reads most often are exposed as attributes on top of it, so there is one source of truth rather than a pair of fields to keep in step.

In [ ]:
#| export
class Chat:
    "Backend-agnostic chat. `Chat(model)` returns the subclass for that model's runtime."
    _runtime = None          # the registry name this subclass serves
    _opt_map = {}            # portable option name -> this backend's own (dotted nests)
    _opt_skip = ()           # ...and the ones it cannot honour at all
    _dflt_cbs = []
    _stream_raw = False      # `stream='raw'` yields chunk dicts instead of markdown
    _media_ok = True         # can this transport carry pictures and sound?
    _media_note = ''         # ...and if not, what to use instead

    def __new__(cls, model=None, *, runtime=None, model_path=None, **kw):
        if cls is not Chat: return super().__new__(cls)
        nm, _ = resolve_runtime(model, runtime, model_path)
        sub = get_runtime(nm)
        # through the subclass's own `__new__`, not `object.__new__`: a backend may re-route
        # again (an MLX vision repo picks a different class)
        return sub.__new__(sub, model, runtime=runtime, model_path=model_path, **kw)

    def __init__(self, model=None, *, runtime=None, model_path=None, opts=None, **kw):
        self._setup(model, ChatOpts.create(opts, **kw))

    def _setup(self, model=None, opts=None):
        "Shared init tail: strip the runtime prefix, store the options, build the history, register callbacks."
        _, model = split_runtime(model)
        o = self.opts = ChatOpts.create(opts)
        if o.max_parallel_tools and o.max_parallel_tools < 1:
            raise ValueError('max_parallel_tools must be at least 1')
        self.tools = L(o.tools)
        self.hist = self.fmt2hist(o.messages)
        self.use, self.cbs, self.turn_msg, self.turn_res = UsageStats(), L(), None, None
        self._steps, self._budget_exceeded, self._final_sent = 0, False, False
        self._cancel = threading.Event()
        if o.default_cbs: self.add_cbs(self._dflt_cbs)
        self.add_cbs(o.cbs)
        return model

    mk_content, mk_msg, mk_msgs = staticmethod(mk_content), staticmethod(mk_msg), staticmethod(mk_msgs)

    def fmt2hist(self, msgs):
        "Backend messages -> canonical urai history dicts. Backends override; the base normalizes."
        return self.mk_msgs(msgs)

    @property
    def runtime(self): return self._runtime
    @property
    def pct_full(self): return self.token_count / self.ctx_limit
    @property
    def cancelled(self): return self._cancel.is_set()

    def map_opts(self, kw):
        "Portable option names under this backend's own, dropping what it cannot honour."
        out = {}
        for k, v in (kw or {}).items():
            if k in self._opt_skip: continue
            set_dotted(out, self._opt_map.get(k, k), v)
        return out

The options a caller reaches for constantly get an attribute apiece, reading and writing straight through `chat.opts`. Setting `chat.sp` therefore changes the options too, and there is no second copy to fall out of date.

In [ ]:
#| export
#: Options common enough to deserve an attribute of their own.
_OPT_ATTRS = ('sp', 'approve', 'max_steps', 'tool_max_len', 'parallel_tools',
              'max_parallel_tools', 'final_prompt', 'ctx')

def _opt_prop(name):
    return property(lambda self: getattr(self.opts, name),
                    lambda self, v: setattr(self, 'opts', replace(self.opts, **{name: v})))

for _n in _OPT_ATTRS: setattr(Chat, _n, _opt_prop(_n))

## Callbacks

A callback can be handed over as a class or as an instance; a class is instantiated here. Removing accepts either too, and removing by class drops every callback of that type — which is how a caller turns off a default it did not ask for.

In [ ]:
#| export
@patch
def add_cb(self:Chat, cb):
    "Register a callback, class or instance. Binds `cb.chat` and returns the instance."
    if isinstance(cb, type): cb = cb()
    cb.chat = self; self.cbs.append(cb); return cb

@patch
def add_cbs(self:Chat, cbs):
    "Register several callbacks and return the `L` of registered instances."
    return L(cbs).map(self.add_cb)

@patch
def remove_cb(self:Chat, cb):
    "Remove a callback by instance, or by class to drop every callback of that type."
    keep = (lambda c: not isinstance(c, cb)) if isinstance(cb, type) else (lambda c: c is not cb)
    self.cbs = self.cbs.filter(keep); return self

@patch
def remove_cbs(self:Chat, cbs):
    "Remove several callbacks, by instance or by class."
    L(cbs).map(self.remove_cb); return self

## One turn

`__call__` resets the per-turn state, hands the message to the backend, and sends one closing round if the tool-call budget ran out mid-turn — otherwise a caller gets a reply that stops at a tool call and says nothing.

Turn options are the other half of the parameter fix. Generation settings for a single call go here and reach `_send` under their portable names, so making one turn think harder no longer means reaching in and mutating an attribute.

In [ ]:
#| export
@patch
def __call__(self:Chat, msg=None, stream=False, cbs=None, **kw):
    '''Run one chat turn: a `Resp`, or a generator when `stream` is set.

    `stream=True` yields markdown strings, ready to print or hand to `display_stream`.
    `stream='raw'` yields the underlying chunk dicts instead, for a consumer that wants the
    text/thinking/tool-call structure rather than rendered markdown.
    `kw` sets generation options (`temp`, `effort`, `max_output_tokens`, ...) for this turn alone.

    One `Chat` is one conversation. `hist`, the turn state and the backend cache are shared
    mutable state, so a single instance is not safe to drive from two threads at once.'''
    self.use, self._steps, self._budget_exceeded, self._final_sent = UsageStats(), 0, False, False
    self._cancel.clear()
    self._stream_raw = stream == 'raw'
    kw = turn_kw(**kw)
    if stream: return self._stream_turn(msg, cbs, **kw)
    added = self.add_cbs(cbs)
    try:
        r = self._send(msg, **kw)
        if self._budget_exceeded and not self._final_sent:
            self._final_sent = True
            r = self._send(self.final_prompt, **kw)
        return r
    finally: self.remove_cbs(added)

@patch
def _stream_turn(self:Chat, msg, cbs=None, **kw):
    "Stream one turn, plus a `final_prompt` round if the tool-call budget ended it early."
    r = yield from self._stream(msg, cbs, **kw)
    if self._budget_exceeded and not self._final_sent:
        self._final_sent = True
        r = yield from self._stream(self.final_prompt, cbs, **kw)
    return r

@patch
def _emit(self:Chat, o, fmt):
    "One streamed chunk: the raw dict in raw mode, else `fmt`-rendered markdown."
    return o if self._stream_raw else fmt.format_item(o)

@patch
def _check_media(self:Chat):
    "Refuse media a text-only transport cannot carry, rather than dropping it in silence."
    if self._media_ok: return
    c = (self.turn_msg or {}).get('content')
    if isinstance(c, list) and any(is_media(p) for p in c): raise TypeError(self._media_note)

### A backend to test against

Everything above works without a model, and so should its tests. `_DemoChat` is the smallest thing that satisfies the backend contract: it records what it was sent and echoes it back.

In [ ]:
class _DemoChat(Chat):
    "A backend that answers without a model, so the machinery around one can be tested."
    _runtime, _opt_map, _opt_skip = 'demo', {'ctx': 'n_ctx'}, ('effort',)
    ctx_limit, token_count = 8192, 0

    def __init__(self, model=None, *, runtime=None, model_path=None, opts=None, **kw):
        self.model_id, self.sent = model, []
        self._setup(model, ChatOpts.create(opts, **kw))

    def _send(self, msg, **kw):
        self.sent.append((msg, kw))
        self.turn_msg = self.mk_msg(msg) if msg is not None else None
        if self.turn_msg: self.hist.append(self.turn_msg)
        self.turn_res = Resp({'role': 'assistant',
                              'content': f'echo: {resp_text(self.turn_msg or {})}'})
        self.hist.append(dict(self.turn_res))
        for _ in run_cbs(self, 'after_response'): pass
        return self.turn_res

    def _stream(self, msg, cbs=None, **kw):
        r = self._send(msg, **kw)
        yield resp_text(r)
        return r

register_runtime(Runtime('demo', _DemoChat, ('demo-',)), dflt=True)

In [ ]:
c = Chat('demo-1', sp='Be brief.', ctx=4096, temp=0.2)
test_eq(type(c).__name__, '_DemoChat')           # dispatched by the id pattern
test_eq(isinstance(c, Chat), True)
test_eq((c.sp, c.ctx, c.runtime), ('Be brief.', 4096, 'demo'))
test_eq(c.opts.temp, 0.2)

In [ ]:
# the settings the loop reads are one thing, not two
c.sp = 'Be terse.'
test_eq((c.sp, c.opts.sp), ('Be terse.', 'Be terse.'))
test_fail(lambda: Chat('demo-1', max_parallel_tools=-1), contains='at least 1')

In [ ]:
r = c('hello')
test_eq(resp_text(r), 'echo: hello')
test_eq([m['role'] for m in c.hist], ['user', 'assistant'])
test_eq(''.join(c('again', stream=True)), 'echo: again')

In [ ]:
# a turn option reaches `_send` without anything being mutated first
c('hi', temp=0.9, effort='high')
test_eq(c.sent[-1][1], {'temp': 0.9, 'effort': 'high'})
test_eq(c.opts.temp, 0.2)                        # the chat's own setting is untouched
test_eq(c.map_opts({'ctx': 8192, 'effort': 'high'}), {'n_ctx': 8192})   # renamed, and skipped

In [ ]:
class _Counter(ChatCallback):
    def __init__(self): self.n = 0
    def after_response(self): self.n += 1

c = Chat('demo-1')
cb = c.add_cb(_Counter)          # a class is instantiated for you
test_eq((isinstance(cb, _Counter), cb.chat is c), (True, True))
c('one'); c('two')
test_eq(cb.n, 2)

In [ ]:
c.remove_cb(_Counter)            # by class: every callback of that type goes
test_eq(len(c.cbs), 0)
c('three')
test_eq(cb.n, 2)                 # ...and it stopped counting

In [ ]:
# callbacks handed to one call are registered for that call only
one_off = _Counter()
c('four', cbs=[one_off])
test_eq((one_off.n, len(c.cbs)), (1, 0))
c('five')
test_eq(one_off.n, 1)

## Reconfiguring

Changing the system prompt or the tool set mid-conversation leaves the backend with state to rebuild: litert holds a `Conversation`, mlx holds a KV cache. `reconfigure` is the door that tells it, which a bare attribute assignment does not.

In [ ]:
#| export
@patch
def _recreate_conv(self:Chat):
    "Rebuild whatever conversation state the backend holds from `self.hist`. A no-op for backends that re-send the whole list."
    pass

@patch
def _set_sp(self:Chat, sp):
    "Note a new system prompt wherever the backend keeps one. `_recreate_conv` applies it."
    pass

@patch
def _set_tools(self:Chat, tools):
    "Rebuild whatever tool state the backend holds: wire schemas, a call namespace."
    pass

@patch
def reconfigure(self:Chat, sp=None, tools=None):
    "Change `sp` or `tools` on a live conversation, keeping `hist`. `None` leaves that half alone; `tools=()` removes every tool."
    if sp is not None: self.sp = sp; self._set_sp(sp)
    if tools is not None:
        self.opts = replace(self.opts, tools=tuple(L(tools)))
        self.tools = L(tools); self._set_tools(self.tools)
    self._recreate_conv()
    return self

In [ ]:
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

c = Chat('demo-1', sp='old')
c('remember this')
c.reconfigure(sp='new', tools=[add])
test_eq((c.sp, c.opts.sp), ('new', 'new'))
test_eq([t.__name__ for t in c.tools], ['add'])
test_eq(len(c.hist), 2)                    # the conversation survived

In [ ]:
c.reconfigure(tools=())                    # an empty tuple means "no tools", not "leave alone"
test_eq(list(c.tools), [])
c.reconfigure(sp='newer')
test_eq((list(c.tools), c.sp), ([], 'newer'))   # tools=None left them alone

## Stopping, and the rest

Cancelling is cooperative, not a kill. The loop stops between chunks and between tool calls, so the history is left with a result for every call and the next turn can still be sent. What that costs is that a backend with no abort seam finishes the completion it is already inside.

In [ ]:
#| export
@patch
def cancel(self:Chat):
    "Ask the turn in flight to stop at its next safe point. Safe to call from another thread."
    self._cancel.set()
    return True

@patch
def oneshot(self:Chat, prompt, sp='', think=None, max_tokens=None):
    "One stateless reply: no history, no tools, nothing kept. `think=False` asks the model not to deliberate."
    return self._oneshot(prompt, sp, think=think, max_tokens=max_tokens)

@patch
def _oneshot(self:Chat, prompt, sp='', think=None, max_tokens=None):
    "One stateless completion's text. Every backend implements it; `oneshot` is the public door."
    raise NotImplementedError

@patch
def recover_context(self:Chat, error, max_output_tokens=None):
    "Backend contract for recovering from a full context window."
    raise ContextWindowExceededError(
        f'context recovery is not implemented for {type(self).__name__}') from error

@patch
def close(self:Chat):
    "Release backend resources. Overridden per backend."
    pass

@patch
def __del__(self:Chat):
    try: self.close()
    except Exception: pass

@patch
def print_hist(self:Chat):
    "Render the conversation history as markdown, or plain text outside a notebook."
    md = '\n\n---\n\n'.join(f"**{m.get('role','?')}**\n\n{Resp(m)._repr_markdown_()}"
                            for m in self.hist)
    try:
        from IPython.display import Markdown, display
        display(Markdown(md))
    except Exception: print(md)

In [ ]:
c = Chat('demo-1')
test_eq(c.cancelled, False)
test_eq(c.cancel(), True)
test_eq(c.cancelled, True)
c('a turn clears it')
test_eq(c.cancelled, False)

In [ ]:
test_fail(lambda: c.oneshot('hi'))         # the demo backend implements neither of these
test_fail(lambda: c.recover_context(ValueError('full')),
          contains='context recovery is not implemented')
test_eq(c.close(), None)

In [ ]:
c.print_hist()

## The factory

`chat()` takes a name, a `ModelSpec`, or nothing. Given a spec it unpacks the window and the runtime options that were resolved alongside it, which is the whole of what a caller used to have to do by hand.

In [ ]:
#| export
def chat(model=None, opts=None, *, runtime=None, model_path=None, **kw):
    "Build a `Chat`. `model` is a name, a `ModelSpec`, or nothing for the default runtime."
    if isinstance(model, ModelSpec):
        spec = model
        model, runtime = spec.model_id, spec.runtime
        kw = {'ctx': spec.ctx, **spec.opts, **kw}
    return Chat(model, runtime=runtime, model_path=model_path, opts=ChatOpts.create(opts, **kw))

In [ ]:
c = chat('demo-1', sp='Be brief.')
test_eq((type(c).__name__, c.sp), ('_DemoChat', 'Be brief.'))

In [ ]:
spec = resolve('demo-1', ctx=2048, temp=0.3)
c = chat(spec)
test_eq((c.ctx, c.opts.temp, c.runtime), (2048, 0.3, 'demo'))
test_eq(chat(spec, temp=0.9).opts.temp, 0.9)      # a call-site option beats the spec's
test_eq(chat().runtime, 'demo')                   # nothing named: the default runtime

## Async

`AsyncChat` wraps a sync `Chat` and runs its blocking calls in a worker thread. An async generator cannot return a value, so a streamed turn hands the final `Resp` back on the iterator's `.value` once it is drained.

In [ ]:
#| export
class AsyncChat(GetAttr):
    "Async twin of `Chat`. Blocking calls run in a worker thread."
    _default = 'chat'
    def __init__(self, model=None, *, runtime=None, **kw):
        self.chat = model if hasattr(model, '_send') else Chat(model, runtime=runtime, **kw)

    async def __call__(self, msg=None, stream=False, cbs=None, **kw):
        "Run one chat turn. `await` the result, or an async chunk iterator when `stream` is set."
        if stream: return self._astream(msg, cbs, **kw)
        return await asyncio.to_thread(lambda: self.chat(msg, False, cbs, **kw))

    @asave_iter
    async def _astream(it, self, msg, cbs=None, **kw):
        "Drive the sync stream in a worker thread. The iterator's `.value` holds the final `Resp`."
        g = SaveReturn(self.chat(msg, stream=True, cbs=cbs, **kw))
        gi, done = iter(g), object()
        while (o := await asyncio.to_thread(next, gi, done)) is not done: yield o
        it.value = g.value

    def close(self): self.chat.close()
    async def __aenter__(self): return self
    async def __aexit__(self, *exc): self.close()

async def adisplay_stream(chunks):
    "Async twin of `display_stream`. Renders live in a notebook, and returns the full markdown."
    if not hasattr(chunks, '__aiter__'): chunks = await chunks
    from IPython.display import display, Markdown
    h, md = display(Markdown(''), display_id=True), ''
    async for c in chunks:
        md += c
        if h is not None: h.update(Markdown(md))
    return md

In [ ]:
a = AsyncChat('demo-1', sp='async')
test_eq(a.sp, 'async')                              # GetAttr forwards to the wrapped chat
test_eq(resp_text(await a('hello')), 'echo: hello')

In [ ]:
it = await a('streamed', stream=True)
test_eq([c async for c in it], ['echo: streamed'])
test_eq(resp_text(it.value), 'echo: streamed')      # the final Resp, once drained

In [ ]:
async with AsyncChat(Chat('demo-1')) as a2:         # an existing Chat is wrapped, not rebuilt
    test_eq(resp_text(await a2('wrapped')), 'echo: wrapped')

## Approving a tool call

An approval policy is one function, `approve(tool_call) -> bool`. `hitl_policy` builds one from per-tool modes: `approved` always runs, `dont_run` never does, and anything else asks.

In [ ]:
#| export
def ask_console(tc):
    "Console y/N approval prompt for a tool call."
    a = tc['function'].get('arguments', {})
    return input(f"Authorize {tc['function']['name']}({a})? [y/N] ").strip().lower() in ('y', 'yes')

def http_approval(url=None, timeout=300):
    "An approval callback that asks a running web UI over HTTP."
    endpoint = (url or os.environ.get('URAI_APPROVAL_URL')
                or 'http://127.0.0.1:5001').rstrip('/') + '/agent/approval/request'
    def ask(tc):
        body = json.dumps({'tool_call': tc, 'timeout': timeout}).encode()
        req = Request(endpoint, data=body, headers={'Content-Type': 'application/json'},
                      method='POST')
        try:
            with urlopen(req, timeout=timeout + 5) as res: reply = json.load(res)
        except Exception as e:
            raise RuntimeError(f'approval request failed at {endpoint}: {e}') from e
        return bool(reply.get('answer', {}).get('answer'))
    return ask

def hitl_policy(modes, ask=None, http=False):
    "An `approve(tool_call)` from per-tool modes. `http` asks a web UI instead of the console."
    ask = http_approval(http if isinstance(http, str) else None) if http else (ask or ask_console)
    def approve(tc):
        mode = modes.get(tc['function']['name'], 'check') if modes else 'check'
        return True if mode == 'approved' else False if mode == 'dont_run' else ask(tc)
    return approve

In [ ]:
asked = []
p = hitl_policy({'ls': 'approved', 'rm': 'dont_run'}, ask=lambda tc: asked.append(tc) or True)
test_eq(p(ToolCall('ls')), True)
test_eq(p(ToolCall('rm')), False)
test_eq(asked, [])                       # neither one had to ask

In [ ]:
test_eq(p(ToolCall('anything_else')), True)
test_eq([t.name for t in asked], ['anything_else'])
test_eq(hitl_policy(None, ask=lambda tc: False)(ToolCall('x')), False)   # no modes: ask about all

In [ ]:
test_fail(lambda: http_approval('http://127.0.0.1:9')(ToolCall('x')),
          contains='approval request failed')

In [ ]:
#| hide
import urai.opts as _o
RUNTIMES.clear(); _o.dflt_runtime = None    # leave the registry as a real backend will find it

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()